# Prediksi harga rumah dengan DVC, MLflow, dan CI/CD

Notebook ini menjadi panduan untuk kode yang sudah dipisahkan ke `src/house_price`. Notebook, CLI, pipeline DVC, dan Docker menggunakan fungsi training yang sama.

DVC melacak dataset, Git menyimpan riwayat metadata dan kode, sedangkan MLflow mencatat eksperimen serta model. Scaler dan Random Forest kini disimpan dalam satu pipeline, sehingga model yang dimuat dapat langsung menerima fitur mentah.

Pilih kernel `.venv` proyek dengan Python 3.13 dan jalankan cell dari atas. Notebook memakai configs/quick.yaml untuk praktik kelas. Grid lengkap tetap tersedia di params.yaml.


## 1. Siapkan environment

Jalankan instalasi dari terminal PowerShell di folder proyek:

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements-dev.txt
```

Dependency didefinisikan di `pyproject.toml`. Instalasi mencakup modul proyek, DVC, MLflow, dan alat pengujian. Setelah instalasi pertama, restart kernel bila import belum terbaca.


In [ ]:
from pathlib import Path
from datetime import datetime
from uuid import uuid4
import os
import subprocess
import sys

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import pandas as pd

from house_price.config import load_config
from house_price.data import FEATURES, TARGET, prepare_data, load_dataset
from house_price.versioning import run_dvc, track_data
from house_price.train import train_model
from house_price.predict import load_model, predict

PROJECT_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "pyproject.toml").is_file()
     and (folder / "src" / "house_price").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Buka notebook dari folder house_price atau notebooks.")
print(f"Proyek: {PROJECT_ROOT}")
print(f"Python kernel: {sys.executable}")


## 2. Pastikan Git dan DVC tersedia

Folder kerja ini sudah diinisialisasi. Cell berikut juga dapat dipakai pada salinan proyek baru: setup hanya dilakukan bila belum ada. Tidak ada commit atau push otomatis.


In [ ]:
git_probe = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], cwd=PROJECT_ROOT,
    capture_output=True, text=True,
)
if git_probe.returncode != 0:
    subprocess.run(["git", "init"], cwd=PROJECT_ROOT, check=True)
    git_root = PROJECT_ROOT
else:
    git_root = Path(git_probe.stdout.strip()).resolve()

if not (PROJECT_ROOT / ".dvc").exists():
    options = () if git_root == PROJECT_ROOT.resolve() else ("--subdir",)
    print(run_dvc(PROJECT_ROOT, "init", *options))
print(run_dvc(PROJECT_ROOT, "--version"))


## 3. Siapkan, tracking, dan baca dataset

CSV disimpan di `data/raw/california_housing.csv`. File yang sudah ada tidak ditimpa. Pada clone baru, dataset awal publik dapat diambil dari sumber dengan from_source=True. Isi hasil unduhan harus cocok dengan checksum DVC; versi modifikasi harus dipulihkan lewat cache atau remote DVC.

`track_data` menjalankan `dvc add`. Setelah itu, `load_dataset` memvalidasi kolom, nilai numerik, dan kecocokan CSV dengan checksum DVC. Jalankan cell ini kembali setelah mengganti versi dataset. SHA-256 dan metadata DVC akan dicatat oleh fungsi training ke MLflow.


In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "california_housing.csv"
prepare_data(DATA_PATH, from_source=True)
print(track_data(PROJECT_ROOT, DATA_PATH))
dataset = load_dataset(DATA_PATH)
df = dataset.frame
print(f"Dataset: {df.shape[0]:,} baris, {df.shape[1]} kolom")
print(f"SHA-256: {dataset.sha256}")
print(dataset.dvc_metadata)
df.head()


## 4. Simpan riwayat dan gunakan versi sebelumnya

Perintah pada bagian ini dijalankan manual di **terminal PowerShell**, dari folder proyek. Bagian training di bawah tetap bisa dijalankan sebelum commit; namun versi dataset belum memiliki riwayat Git sampai commit dibuat.

### Simpan versi pertama

```powershell
Set-Location 'F:\programming\mylearning\machine learning\house_price'
git add .dvc .dvcignore .gitignore data/raw/.gitignore data/raw/california_housing.csv.dvc src tests scripts notebooks .github Dockerfile .dockerignore pyproject.toml requirements.txt requirements-dev.txt params.yaml configs dvc.yaml docs README.md
git diff --cached --stat
git commit -m "Track California Housing dataset with DVC"
git tag data-v1
```

Jalankan tag `data-v1` sekali untuk versi pertama. Jika Git meminta identitas, isi `git config user.name` dan `git config user.email` dengan identitas Anda, lalu ulangi commit. Jangan menambahkan CSV atau folder `.dvc/cache` ke Git.

### Catat perubahan dataset

Setelah ada perubahan nyata pada CSV, jalankan langkah 3 lagi agar DataFrame dan metadata DVC ikut diperbarui. Kemudian:

```powershell
git add data/raw/california_housing.csv.dvc
git commit -m "Update California Housing dataset"
git log --oneline -- data/raw/california_housing.csv.dvc
```

Setiap commit metadata menyimpan rujukan ke versi data yang sesuai. `dvc status` memeriksa apakah CSV cocok dengan metadata saat ini.

### Gunakan kembali versi pertama

Simpan perubahan dataset yang ingin dipertahankan dengan `dvc add` dan commit lebih dulu. Perintah berikut mengganti CSV kerja dengan versi `data-v1`, tanpa memindahkan seluruh notebook ke commit lama:

```powershell
git restore --source=data-v1 -- data/raw/california_housing.csv.dvc
.\.venv\Scripts\python.exe -m dvc checkout data/raw/california_housing.csv.dvc
```

Untuk kembali ke versi data pada commit terbaru di branch aktif:

```powershell
git restore --source=HEAD -- data/raw/california_housing.csv.dvc
.\.venv\Scripts\python.exe -m dvc checkout data/raw/california_housing.csv.dvc
```

Setelah mengganti versi, jalankan ulang langkah 3, konfigurasi, training, dan evaluasi. `df`, `X_train`, dan `X_test` yang sudah ada di memori tidak otomatis ikut berubah.

### Opsional: backup data lewat remote DVC

Cache saat ini hanya ada di komputer ini. Untuk backup, siapkan folder backup atau storage yang sesuai, lalu ganti path contoh berikut. Konfigurasi ini tidak dijalankan otomatis:

```powershell
.\.venv\Scripts\python.exe -m dvc remote add -d backup 'D:\backup\house-price-dvc'
.\.venv\Scripts\python.exe -m dvc push data/raw/california_housing.csv.dvc
git add .dvc/config
git commit -m "Configure DVC backup storage"
```

Setelah remote dikonfigurasi dan versi datanya pernah di-push, gunakan `dvc pull data/raw/california_housing.csv.dvc` untuk mengunduh dan memulihkan data ketika cache lokal belum tersedia. Push setiap versi yang ingin disimpan di backup. Git menyimpan metadata, sedangkan remote DVC menyimpan isi dataset.

Referensi: [memulai DVC](https://doc.dvc.org/start), [dvc add](https://doc.dvc.org/command-reference/add), dan [dvc checkout](https://doc.dvc.org/command-reference/checkout).


## 5. Baca konfigurasi training

Notebook memakai `configs/quick.yaml`: dua kandidat dan dua fold, dengan seluruh dataset. CLI dapat memakai file yang sama melalui `--config configs/quick.yaml`. Grid lengkap dan default `dvc repro` berada di `params.yaml`. `random_state=42` menetapkan pembagian data yang konsisten.

Preprocessing berada di dalam pipeline GridSearchCV. Scaler hanya belajar dari bagian training masing-masing fold. Model terbaik dipilih berdasarkan MSE cross-validation; test set dipakai untuk evaluasi akhir.


In [ ]:
config = load_config(PROJECT_ROOT / "configs" / "quick.yaml")
config


## 6. Training dan simpan ke MLflow

Notebook memakai server `http://127.0.0.1:5000`. Sebelum training, jalankan perintah ini di terminal PowerShell kedua dari root proyek:

```powershell
New-Item -ItemType Directory -Force .mlflow/server
.\.venv\Scripts\python.exe -m mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///.mlflow/server/mlflow.db --artifacts-destination .mlflow/server/artifacts --serve-artifacts
```

Biarkan terminal server berjalan. Buka `http://127.0.0.1:5000` untuk melihat run. Environment variable `MLFLOW_TRACKING_URI` dapat dipakai bila sengaja memilih server lain.

Setiap eksekusi membuat folder output baru di `artifacts/notebook`. Di dalamnya ada model lengkap, `metrics.json`, dan `run.json`. Dataset dibaca ulang oleh fungsi training; metadata DVC yang tidak cocok akan ditolak.


In [ ]:
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
run_name = datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "notebook" / run_name

result = train_model(DATA_PATH, OUTPUT_DIR, config, tracking_uri=TRACKING_URI)
print(f"Model URI: {result['model_uri']}")
print(f"Model lokal: {result['local_model_path']}")
print(f"Parameter terbaik: {result['best_params']}")
pd.Series(result["metrics"], name="nilai")


## 7. Load model terbaik dari MLflow

URI hasil `train_model` menunjuk ke pipeline terbaik dari run tersebut. Scaler ikut dimuat, sehingga data input tetap dalam bentuk mentah.

Setelah restart kernel, baca `run.json` dari folder run yang dipilih, lalu gunakan nilai `model_uri` dengan tracking URI yang sama. Alternatif untuk pemakaian tanpa tracking server: load langsung folder `model` yang berada di samping `run.json`.


In [ ]:
loaded_best_model = load_model(result["model_uri"], tracking_uri=TRACKING_URI)
print(f"Model dimuat dari: {result['model_uri']}")


## 8. Prediksi dan evaluasi pada test set yang sama

Dataset dibaca kembali dan checksum dibandingkan dengan hasil training. Pembagian test set mengikuti konfigurasi run. Jangan melakukan scaling manual karena sudah menjadi bagian dari model.


In [ ]:
evaluation_data = load_dataset(DATA_PATH)
if evaluation_data.sha256 != result["dataset_sha256"]:
    raise RuntimeError("Dataset telah berganti versi sejak training. Pulihkan versi run tersebut.")

_, X_test, _, y_test = train_test_split(
    evaluation_data.frame[FEATURES], evaluation_data.frame[TARGET],
    test_size=config.test_size, random_state=config.random_state,
)
y_pred_loaded = predict(loaded_best_model, X_test)["prediction"]
print(f"MSE: {mean_squared_error(y_test, y_pred_loaded):.4f}")
print(f"R2 : {r2_score(y_test, y_pred_loaded):.4f}")
pd.DataFrame({"Aktual": y_test, "Prediksi": y_pred_loaded}).head(10)


## 9. Jalankan alur yang sama tanpa notebook

Dari terminal PowerShell di folder proyek:

```powershell
.\.venv\Scripts\python.exe -m house_price train --config configs/quick.yaml --output artifacts/cli-run-01
.\.venv\Scripts\python.exe -m house_price predict --model-uri artifacts/cli-run-01/model --input data/raw/california_housing.csv --output predictions.csv
```

Untuk training yang dikelola DVC, gunakan `python -m dvc repro`. Stage di `dvc.yaml` bergantung pada kode, CSV yang dilacak DVC, dan `params.yaml`. Setelah berhasil, commit `dvc.lock` bersama perubahan terkait.

GitHub Actions di `.github/workflows/ci-cd.yml` menjalankan lint, pengujian Ubuntu/Windows, validasi notebook, dan pengujian container. Setelah lolos, push ke `main`/`master` atau tag `v*` mempublikasikan Docker image ke `ghcr.io/<owner>/<repository>`. Pull request tidak mempublikasikan image.

Image berisi aplikasi training/prediksi. Dataset dan model tetap diberikan melalui mount atau MLflow. Workflow tidak melakukan training grid penuh pada setiap push. Lihat `README.md` untuk perintah Docker, konfigurasi GitHub, dan cara melihat hasil di GHCR.

Panduan kelas lengkap ada di `docs/PANDUAN_MENGAJAR.md`; lembar latihan ada di `docs/LATIHAN_PESERTA.md`.
